# 05 Visualisasi Hasil & Explainable AI (Results & XAI)

---
### 1. Setup Lingkungan & Import Modul


In [1]:
import sys, joblib
from pathlib import Path

ROOT_DIR = Path(r"c:\Project\napasloka\napasloka")

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

---
### 2. Memuat Hasil Prediksi Data Uji dari Artefak


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import shap

from src.models.interpretation import calculate_tree_shap, mean_absolute_shap
from src.features.scenarios import resolve_scenario


STATION_ID = "pantai-makmur"
POLLUTANT = "co2"
SCENARIO = "scenario_2"

art_base = ROOT_DIR / "artifacts" / "experiments" / STATION_ID / POLLUTANT / SCENARIO
df_pred_xgb = pd.read_csv(art_base / "xgboost" / "predictions.csv")
df_pred_lgb = pd.read_csv(art_base / "lightgbm" / "predictions.csv")
df_pred_xgb["datetime"] = pd.to_datetime(df_pred_xgb["datetime"])

print(f"Total sampel evaluasi data uji: {len(df_pred_xgb):,} jam")
df_pred_lgb.head()


Total sampel evaluasi data uji: 3,109 jam


,datetime,actual,predicted,residual
0,2026-03-28 11:00:00+07:00,442.0,442.779252,-0.779252
1,2026-03-28 12:00:00+07:00,441.0,441.925510,-0.925510
2,2026-03-28 13:00:00+07:00,441.0,441.031336,-0.031336
3,2026-03-28 14:00:00+07:00,440.0,441.691744,-1.691744
4,2026-03-28 15:00:00+07:00,439.0,440.276771,-1.276771


---
### 3. Visualisasi Kurva Deret Waktu (1 Minggu Terakhir Data Uji)


In [ ]:
viz_df_xgb = df_pred_xgb.tail(168)
viz_df_lgb = df_pred_lgb.tail(168)

plt.figure(figsize=(15, 6))
plt.plot(viz_df_xgb["datetime"], viz_df_xgb["actual"], label="Aktual (Ground Truth)", color="#1e293b", linewidth=2.5)
plt.plot(viz_df_xgb["datetime"], viz_df_xgb["predicted"], label="XGBoost Prediksi", color="#ea580c", linestyle="--", alpha=0.85)
plt.plot(viz_df_xgb["datetime"], viz_df_lgb["predicted"], label="LightGBM Prediksi", color="#0284c7", linestyle="-.", alpha=0.85)

plt.title(f"Proyeksi Konsentrasi CO₂ 1 Jam ke Depan vs Aktual (Pantai Makmur, 1 Minggu Terakhir Data Uji)", fontsize=13, pad=12)
plt.xlabel("Waktu Observasi (WIB)", fontsize=11)
plt.ylabel("Konsentrasi CO₂ (ppm)", fontsize=11)
plt.legend(frameon=True, facecolor="#f8fafc", loc="upper left")
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


---
### 4. Analisis Sebaran Galat (Residual Error Distribution)


In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(df_pred_lgb["residual"], bins=40, color="#0284c7", alpha=0.7, edgecolor="black", label="LightGBM Residuals")
plt.axvline(0, color="red", linestyle="--", linewidth=1.5, label="Zero Error")
plt.title("Distribusi Residual Prediksi (Actual - Predicted) LightGBM", fontsize=12)
plt.xlabel("Residual Error (ppm)")
plt.ylabel("Frekuensi")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


---
### 5. Interpretasi Explainable AI: TreeSHAP Beeswarm Plot


In [ ]:
# Muat model terbaik LightGBM
best_model = joblib.load(art_base / "lightgbm" / "model.joblib")

# Muat data uji
scenario_def = resolve_scenario(POLLUTANT, SCENARIO)
feature_cols = list(scenario_def.feature_names)

df_raw = pd.read_csv(ROOT_DIR / "data" / "processed" / f"jabodetabek_{POLLUTANT}_dataset.csv")
df_raw["datetime"] = pd.to_datetime(df_raw["datetime"], utc=True).dt.tz_convert("Asia/Jakarta")
coord_id = "-6.1_107.0"
df_station = df_raw[(df_raw["location"] == coord_id) | (df_raw["location"] == STATION_ID)].sort_values("datetime").reset_index(drop=True)

from src.features.lag_features import create_lag_features
from src.data.clean_data import chronological_train_test_split

df_feat = create_lag_features(df_station, pollutant=POLLUTANT, scenario=SCENARIO, drop_incomplete_rows=True)
_, test_df = chronological_train_test_split(df_feat, test_fraction=0.2)
x_test = test_df[feature_cols]

# Hitung TreeSHAP
explainer = shap.TreeExplainer(best_model)
shap_vals = explainer.shap_values(x_test)

plt.figure(figsize=(10, 6))
plt.title("SHAP Beeswarm Summary Plot - Kontribusi Variabel terhadap Prediksi CO₂", fontsize=12, pad=10)
shap.summary_plot(shap_vals, x_test, show=False)
plt.tight_layout()
plt.show()


---
### 6. Peringkat Kepentingan Fitur Global (Mean |SHAP|)


In [ ]:
df_shap_global = mean_absolute_shap(shap_vals, feature_cols)
print("=== TOP 10 FITUR PALING SIGNIFIKAN MEMPENGARUHI PREDIKSI ===")
display(df_shap_global.head(10))
